# Visualización de Bounding Boxes 3D (Predicción vs Ground Truth)

Este notebook visualiza los emparejamientos y el estado de la segmentación utilizando `plotly`.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go

# Rutas a los archivos
instances_path = 'outputs/Replica/room0/20260427225701-2d8f5fdd/instances.json'
info_path = '../../datasets/replica_v1/room_0/habitat/info_semantic.json'

with open(instances_path, 'r') as f:
    instances_data = json.load(f)
    
with open(info_path, 'r') as f:
    info_data = json.load(f)
    
gt_objects = info_data['objects']
predictions = instances_data['instances']

In [ ]:
def get_box_edges(center, size):
    """Devuelve las coordenadas de las líneas que forman una caja 3D."""
    c = np.array(center)
    s = np.array(size) / 2
    
    # 8 esquinas de la caja
    x_corners = [c[0]-s[0], c[0]+s[0], c[0]+s[0], c[0]-s[0], c[0]-s[0], c[0]+s[0], c[0]+s[0], c[0]-s[0]]
    y_corners = [c[1]-s[1], c[1]-s[1], c[1]+s[1], c[1]+s[1], c[1]-s[1], c[1]-s[1], c[1]+s[1], c[1]+s[1]]
    z_corners = [c[2]-s[2], c[2]-s[2], c[2]-s[2], c[2]-s[2], c[2]+s[2], c[2]+s[2], c[2]+s[2], c[2]+s[2]]
    
    # Indices para dibujar las 12 aristas
    i = [0, 1, 1, 2, 2, 3, 3, 0, 4, 5, 5, 6, 6, 7, 7, 4, 0, 4, 1, 5, 2, 6, 3, 7]
    x_lines, y_lines, z_lines = [], [], []
    
    for idx in range(0, len(i), 2):
        x_lines.extend([x_corners[i[idx]], x_corners[i[idx+1]], None])
        y_lines.extend([y_corners[i[idx]], y_corners[i[idx+1]], None])
        z_lines.extend([z_corners[i[idx]], z_corners[i[idx+1]], None])
        
    return x_lines, y_lines, z_lines

def create_box_trace(center, size, name, color, is_gt=False):
    x, y, z = get_box_edges(center, size)
    line_width = 3 if is_gt else 2
    opacity = 0.4 if is_gt else 1.0
    return go.Scatter3d(
        x=x, y=y, z=z,
        mode='lines',
        name=name,
        line=dict(color=color, width=line_width),
        opacity=opacity,
        hoverinfo='name'
    )

In [ ]:
fig = go.Figure()

# 1. Añadir el Ground Truth (Azul translúcido)
for obj in gt_objects:
    center = obj['oriented_bbox']['abb']['center']
    size = obj['oriented_bbox']['abb']['sizes']
    name = f"GT: {obj['class_name']} (ID:{obj['id']})"
    fig.add_trace(create_box_trace(center, size, name, 'rgba(0, 0, 255, 0.5)', is_gt=True))

# 2. Añadir Predicciones categorizadas por color
color_map = {
    "good-match": 'green',         # Encaja bien
    "over-segmented": 'orange',    # Solo es un trozo
    "under-segmented": 'purple',   # Engulle a varios
    "partial-match": 'yellow',     # Solape menor
    "no-match": 'red'              # Artefactos / Sin pareja
}

for key, val in predictions.items():
    center = val['bbox']['center']
    size = val['bbox']['size']
    
    validation = val.get('validation', {})
    status = validation.get('segmentation_status', 'no-match')
    gt_class = validation.get('matched_gt_class', 'None')
    
    color = color_map.get(status, 'gray')
    name = f"Pred: {key} [{status}] -> {gt_class}"
    
    fig.add_trace(create_box_trace(center, size, name, color, is_gt=False))

# 3. Añadir trayectoria de cámaras
cameras_path = 'outputs/Replica/room0/20260427225701-2d8f5fdd/cameras.json'
try:
    with open(cameras_path, 'r') as f:
        cameras_data = json.load(f)
    cam_x = [c['position'][0] for c in cameras_data]
    cam_y = [c['position'][1] for c in cameras_data]
    cam_z = [c['position'][2] for c in cameras_data]
    fig.add_trace(go.Scatter3d(
        x=cam_x, y=cam_y, z=cam_z,
        mode='markers+lines',
        name='Camera Trajectory',
        marker=dict(size=3, color='black'),
        line=dict(color='black', width=2),
        opacity=0.7
    ))
except Exception as e:
    print(f"No se pudieron cargar las cámaras: {e}")

# Configurar el layout interactivo
fig.update_layout(
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data' # Mantener proporciones reales
    ),
    title="Visualización de Evaluación 3D (Ground Truth vs Predicciones)",
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0, y=1, traceorder="normal", font=dict(family="sans-serif", size=10, color="black"), bgcolor="LightSteelBlue", bordercolor="Black", borderwidth=2)
)

fig.show()